# Day A7 — Bioconductor 분석 맛보기와 재현성: 선형 모델, limma·DESeq2의 발상, sessionInfo, renv

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/A-r-bioconductor/day-A7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/A-r-bioconductor/day-A7.md)

In [ ]:
# Colab 기본 R 런타임에 없는 Bioconductor 패키지 설치 (몇 분 걸립니다)
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("DESeq2", "limma", "SummarizedExperiment"), update = FALSE, ask = FALSE)

## 예시로 이해하기

> 모든 값은 연습용 **가상 값**입니다. 유전자 300개, 샘플 6개(군당 3), 배치 2개이며, 앞 30개 유전자만 처리군에서 log2 값이 1.5 올라가도록 만들었습니다. 정규화를 마친 log 척도 값이라고 가정합니다. 아래 코드를 `a7_data.R`로 저장해 예시 1–2가 불러옵니다.

In [ ]:
# 레슨처럼 아래 코드를 a7_data.R 파일로 저장한 뒤 불러온다 (뒤 셀들이 source()로 다시 부른다)
writeLines(r"---(
# a7_data.R
set.seed(2024)
n_genes <- 300; n_per <- 3
meta <- data.frame(sample = sprintf("S%d", 1:6),
                   group = factor(rep(c("ctrl", "treat"), each = n_per), levels = c("ctrl", "treat")),
                   batch = factor(c("b1", "b2", "b1", "b2", "b1", "b2")))
gene_sd <- sqrt(0.05 * 3 / rchisq(n_genes, df = 3))          # 유전자마다 다른 잡음 크기
truth <- seq_len(n_genes) <= 30                               # 앞 30개만 진짜 변화
y <- matrix(rnorm(n_genes * 6, mean = 8, sd = rep(gene_sd, 6)), nrow = n_genes,
            dimnames = list(sprintf("gene%03d", 1:n_genes), meta$sample))
y[truth, meta$group == "treat"] <- y[truth, meta$group == "treat"] + 1.5
y[, meta$batch == "b2"] <- y[, meta$batch == "b2"] + 0.3      # 배치 효과
)---", "a7_data.R")
source("a7_data.R")

### 예시 1 — 설계 행렬과 유전자별 선형 모델

In [ ]:
source("a7_data.R")
design <- model.matrix(~ batch + group, data = meta)
design
fit_one <- function(v) {
  co <- summary(lm(v ~ batch + group, data = meta))$coefficients
  c(log2FC = co["grouptreat", "Estimate"], p = co["grouptreat", "Pr(>|t|)"])
}
res <- as.data.frame(t(apply(y, 1, fit_one)))
res$padj <- p.adjust(res$p, method = "BH")
head(res[order(res$p), ], 5)
c(raw_p = sum(res$p < 0.05), BH = sum(res$padj < 0.05),
  BH_true = sum(res$padj < 0.05 & truth))

### 예시 2 — limma의 발상을 손으로: 분산 당기기

In [ ]:
source("a7_data.R")
X <- model.matrix(~ batch + group, data = meta)
fit <- lm.fit(X, t(y))                                   # 모든 유전자를 한 번에 적합
df_res <- nrow(X) - ncol(X)
s2 <- colSums(fit$residuals^2) / df_res                  # 유전자별 잔차 분산
XtXinv <- solve(crossprod(X))
beta <- fit$coefficients["grouptreat", ]
se_unit <- sqrt(XtXinv["grouptreat", "grouptreat"])
t_ord <- beta / (sqrt(s2) * se_unit)
p_ord <- 2 * pt(-abs(t_ord), df_res)
# shortcut: d0와 s0^2를 고정값으로 둔다. limma는 이 둘을 데이터에서 추정한다(eBayes)
d0 <- 4; s0sq <- median(s2)
s2_mod <- (d0 * s0sq + df_res * s2) / (d0 + df_res)     # 유전자 분산을 공통값 쪽으로 당김
t_mod <- beta / (sqrt(s2_mod) * se_unit)
p_mod <- 2 * pt(-abs(t_mod), df_res + d0)
truth <- seq_len(nrow(y)) <= 30
summ <- function(p) { hit <- p.adjust(p, "BH") < 0.05; c(selected = sum(hit), true = sum(hit & truth), false = sum(hit & !truth)) }
rbind(ordinary_t = summ(p_ord), moderated_t = summ(p_mod))
smallest <- order(s2)[1:3]
round(cbind(s2 = s2[smallest], s2_mod = s2_mod[smallest]), 4)
stopifnot(all(abs(beta - apply(y, 1, function(v) coef(lm(v ~ batch + group, data = meta))["grouptreat"])) < 1e-8))

### 예시 3 — 실제 도구의 모양: limma와 DESeq2

> 설치: `BiocManager::install(c("limma", "DESeq2"))`. 아래 출력은 R 4.3.2 + Bioconductor 3.18(limma, DESeq2 1.42.1)에서 실행한 그대로입니다. 함수 이름과 인자는 각 패키지의 공식 문서([limma](https://bioconductor.org/packages/release/bioc/html/limma.html), [DESeq2](https://bioconductor.org/packages/release/bioc/html/DESeq2.html))에서 설치한 버전에 맞춰 확인하세요.

In [ ]:
# (가) limma: 정규화된 log 척도 행렬 y와 meta가 있을 때 (예시 1–2의 진짜 도구판)
library(limma)
design <- model.matrix(~ batch + group, data = meta)
fit <- lmFit(y, design)
fit <- eBayes(fit)                       # 예시 2의 '분산 당기기'를 데이터로부터 추정
topTable(fit, coef = "grouptreat", number = 10, adjust.method = "BH")

# (나) DESeq2: RNA-seq의 raw count 정수 행렬일 때만. nCounter 정규화 값에는 쓰지 않는다
library(DESeq2)
set.seed(7)                                   # 연습용 가상 RNA-seq count (y와 같은 크기·같은 진짜 변화)
mu <- matrix(200, nrow(y), ncol(y), dimnames = dimnames(y))
mu[truth, meta$group == "treat"] <- 200 * 2^1.5
raw_counts <- matrix(rnbinom(length(mu), mu = mu, size = 10), nrow(y), dimnames = dimnames(y))
dds <- DESeqDataSetFromMatrix(countData = raw_counts, colData = meta, design = ~ batch + group)
dds <- DESeq(dds, quiet = TRUE)
res <- results(dds, contrast = c("group", "treat", "ctrl"))
head(res[order(res$pvalue), ], 5)
c(BH = sum(res$padj < 0.05, na.rm = TRUE), BH_true = sum(res$padj < 0.05 & truth, na.rm = TRUE))
is(dds, "SummarizedExperiment")

### 예시 4 — 환경 기록 남기기

In [ ]:
suppressPackageStartupMessages(library(dplyr))
si <- sessionInfo()
si$R.version$version.string
names(si$otherPkgs)
dir.create("reports/advanced/A", recursive = TRUE, showWarnings = FALSE)
writeLines(capture.output(si), "reports/advanced/A/sessionInfo.txt")
length(readLines("reports/advanced/A/sessionInfo.txt")) > 5
proj <- file.path(tempdir(), "demo_proj"); dir.create(proj, showWarnings = FALSE)
writeLines(c("library(dplyr)", "library(ggplot2)", "x <- stats::median(1:3)"), file.path(proj, "analysis.R"))
deps <- renv::dependencies(proj, progress = FALSE)
sort(unique(deps$Package))

renv를 실제 프로젝트에 적용하는 명령은 아래와 같습니다. 이 명령은 프로젝트 폴더에 `renv/`와 `renv.lock`을 만들고 패키지를 프로젝트 라이브러리로 설치하므로, 연습은 저장소가 아닌 별도의 빈 폴더에서 하세요. 이 레슨을 쓸 때 빈 연습 폴더(R 파일 하나)에서 네 명령을 차례로 실행해 `renv/`, `renv.lock`, `.Rprofile`이 생기고 `renv::status()`가 동기화됨(synchronized)으로 나오는 것을 확인했습니다. 스크립트에서 실행할 때는 `renv::init(restart = FALSE)`, `renv::snapshot(prompt = FALSE)`처럼 확인 질문을 끄는 인자를 붙입니다.

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# renv::init()       # 프로젝트 전용 라이브러리를 만들고 지금 쓰는 패키지를 기록
# renv::snapshot()   # 패키지를 추가·변경한 뒤 renv.lock 갱신
# renv::status()     # 코드와 lock 파일이 어긋난 곳 확인
# renv::restore()    # 다른 컴퓨터에서 lock 파일대로 설치